# New

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np

# 預測不同階段的平均成績

In [15]:
export_name = 'data_0423b.csv'

studentInfo = pd.read_csv('studentInfo.csv')
studentAssess = pd.read_csv('studentAssessment.csv')
assessments = pd.read_csv('assessments.csv')
studentRegistration = pd.read_csv('studentRegistration.csv')
course = pd.read_csv('courses.csv')

df = pd.merge(studentAssess, assessments, on='id_assessment')

def label_stage(row, course_length):
    if row['date'] <= course_length * 0.25:
        return 'early'
    elif row['date'] <= course_length * 0.75:
        return 'mid'
    else:
        return 'late'

# 合併回學生資料並標記階段
df = pd.merge(df, course, on=['code_module', 'code_presentation'], how='left')
df['stage'] = df.apply(lambda row: label_stage(row, row['module_presentation_length']), axis=1)

# 計算每個階段的平均成績
stage_scores = df.pivot_table(index=['id_student', 'code_module', 'code_presentation'],
                              columns='stage', values='score', aggfunc='mean').reset_index()
stage_scores.rename(columns={'early': 'early_score', 'mid': 'mid_score', 'late': 'late_score'}, inplace=True)

#### SCORE_DIFF #####
stage_scores['score_diff'] = stage_scores['late_score'] - stage_scores['early_score']
stage_scores = stage_scores.dropna(subset=['score_diff'])

merged_df = pd.merge(stage_scores, studentInfo, on=['code_module', 'code_presentation', 'id_student'], how='left')
merged_df = pd.merge(merged_df, studentRegistration, on=['code_module', 'code_presentation', 'id_student'], how='left')

print(merged_df.shape)
merged_df.head()

(15961, 18)


,id_student,code_module,code_presentation,early_score,late_score,mid_score,score_diff,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
0,6516,AAA,2014J,54.000000,77.000000,62.000000,23.000000,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
1,11391,AAA,2013J,81.500000,82.000000,82.500000,0.500000,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN
2,23698,CCC,2014J,88.333333,67.666667,67.333333,-20.666667,F,East Anglian Region,A Level or Equivalent,50-60%,0-35,0,120,N,Pass,-110.0,NaN
3,23798,BBB,2013J,93.333333,94.500000,94.000000,1.166667,M,Wales,A Level or Equivalent,50-60%,0-35,0,60,N,Distinction,-27.0,NaN
4,24213,DDD,2014B,84.500000,61.000000,81.000000,-23.500000,F,East Anglian Region,A Level or Equivalent,40-50%,0-35,1,60,N,Pass,-54.0,NaN


# Normalization

In [16]:
def normalize(df):
    df = merged_df.copy()
    # df = pd.read_csv('dataMerged.csv')

    # 找出非 nominal 欄位（數值欄位，但排除 nominal 和 target 欄位，如果你有一個叫 'score' 的 target 欄位）
    non_nominal_cols = df.select_dtypes(include=['number']).columns.difference(['score', 'avg_score', 'completion_ratio', 'id_student', 'score_diff'])

    # 初始化 scaler
    scaler = MinMaxScaler()

    # 對非 nominal 欄位做 normalization，並直接覆蓋原欄位
    df[non_nominal_cols] = scaler.fit_transform(df[non_nominal_cols])

    print(df.describe())
    return df

# Assessments Stats

In [17]:
# 讀檔
assessments = pd.read_csv('assessments.csv')
student_assessments = pd.read_csv('studentAssessment.csv')

# 篩出非考試類型的作業
assignments_only = assessments[assessments['assessment_type'] != 'Exam']

# 建立學生與作業的 Cartesian join（所有學生該門課應該繳交的所有作業）
expected = pd.merge(
    student_assessments[['id_student']].drop_duplicates(),
    assignments_only,
    how='cross'  # 或改用自己控制範圍的 merge
)

# 合併實際繳交資料（左邊保留所有應繳交作業）
full_data = pd.merge(
    assignments_only,
    student_assessments,
    on='id_assessment',
    how='left',
    suffixes=('_assessment', '_student')
)

# 計算遲交欄位：僅當有繳交時再判斷
full_data['is_late'] = (full_data['date_submitted'] > full_data['date']) & full_data['date_submitted'].notna()

# 補上沒繳交的學生：用 assessments 去對 studentAssessment 的學生和作業做 Cartesian join
students = student_assessments[['id_student']].drop_duplicates()
assignment_students = pd.merge(assignments_only, students, how='cross')

# 合併應繳資料與實際繳交情況（左邊是所有應該出現的 student-assignment 組合）
complete = pd.merge(
    assignment_students,
    full_data[['id_assessment', 'id_student', 'score', 'is_late']],
    on=['id_assessment', 'id_student'],
    how='left'
)

# 最終彙總：每位學生在每門課的資料
ass_stats = complete.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    total_submissions=('score', lambda x: x.notna().sum()),
    total_late=('is_late', lambda x: x.fillna(False).sum()),
    avg_score=('score', 'mean'),
    expected_assignments=('id_assessment', 'count')
).reset_index()

# 補上完成比例
ass_stats['late_ratio'] = ass_stats['total_late'] / ass_stats['total_submissions']
ass_stats['completion_ratio'] = ass_stats['total_submissions'] / ass_stats['expected_assignments']
ass_stats.drop(columns=['expected_assignments', 'total_late', 'total_submissions'], inplace=True)

# 顯示
ass_stats.head()


/var/folders/_q/xfvw_6_d3c5b8t1lpm7rppnh0000gn/T/ipykernel_7823/211209377.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  total_late=('is_late', lambda x: x.fillna(False).sum()),


,id_student,code_module,code_presentation,avg_score,late_ratio,completion_ratio
0,6516,AAA,2013J,NaN,NaN,0.0
1,6516,AAA,2014J,61.8,0.0,1.0
2,6516,BBB,2013B,NaN,NaN,0.0
3,6516,BBB,2013J,NaN,NaN,0.0
4,6516,BBB,2014B,NaN,NaN,0.0


In [18]:
merged_df = pd.merge(merged_df, ass_stats, on=['code_module', 'code_presentation', 'id_student'], how='left')
print(merged_df.shape)

(15961, 21)


# Feature Fusion on Click Stats

In [19]:
activity_to_category = {
    'homepage': 'LPB',
    'page': 'LPB',
    'subpage': 'LPB',
    'sharedsubpage': 'LPB',
    'resource': 'KAB',
    'oucontent': 'KAB',
    'htmlactivity': 'KAB',
    'url': 'KAB',
    'glossary': 'KAB',
    'folder': 'KAB',
    'dataplus': 'KAB',
    'ouwiki': 'KAB',
    'dualpane': 'KAB',
    'forumng': 'ILB',
    'oucollaborate': 'ILB',
    'ouelluminate': 'ILB',
    'quiz': 'LCB',
    'externalquiz': 'LCB',
    'questionnaire': 'LCB',
    'repeatactivity': 'LCB',
}

## 不同學習行為類型的分佈

In [20]:
# 上傳必要的檔案
df_path = "./studentVle.csv"
vle_path = "./vle.csv"

# 讀取資料
df = pd.read_csv(df_path)
vle = pd.read_csv(vle_path)

# 合併 activity_type
df = pd.merge(df, vle[['id_site', 'activity_type']], on='id_site', how='left')

# 加入分類欄位
df['category'] = df['activity_type'].map(activity_to_category)

# 結果表格初始化
final_results = []

# 對四大類別 + 全部做統計
for cat in ['ALL', 'LPB', 'KAB', 'ILB', 'LCB']:
    if cat == 'ALL':
        subset = df.copy()
    else:
        subset = df[df['category'] == cat]
    
    # 按日期合併點擊數
    df_sorted = subset.sort_values(by=['code_module', 'code_presentation', 'id_student', 'id_site', 'date'])
    df_merged = df_sorted.groupby(['code_module', 'code_presentation', 'id_student', 'date']).agg(
        sum_click=('sum_click', 'sum')
    ).reset_index()

    # 日期間隔計算
    df_merged['date_diff'] = df_merged.groupby(['code_module', 'code_presentation', 'id_student'])['date'].diff()

    # 間隔統計
    interval_stats = df_merged.groupby(['code_module', 'code_presentation', 'id_student']).agg(
        mean_click_interval=('date_diff', 'mean'),
        std_click_interval=('date_diff', 'std'),
        mean_clicks=('sum_click', 'mean')
    ).reset_index()

    # 建立完整日期範圍
    def create_full_date_range(group):
        full_range = pd.DataFrame({'date': range(group['date'].min(), group['date'].max() + 1)})
        return full_range.merge(group, on='date', how='left').fillna({'sum_click': 0})

    df_full = df_merged.groupby(['code_module', 'code_presentation', 'id_student']).apply(create_full_date_range).reset_index(drop=True)

    # 變異係數計算
    df_full['click_variance'] = df_full.groupby(['code_module', 'code_presentation', 'id_student'])['sum_click'].transform(
        lambda x: np.std(x) / np.mean(x) if np.mean(x) != 0 else 0
    )

    variance_stats = df_full.groupby(['code_module', 'code_presentation', 'id_student']).agg(
        click_variance=('click_variance', 'first')
    ).reset_index()

    # 合併所有統計結果
    summary = pd.merge(interval_stats, variance_stats, on=['code_module', 'code_presentation', 'id_student'])
    summary.columns = ['code_module', 'code_presentation', 'id_student'] + [f'{cat.lower()}_{col}' for col in summary.columns[3:]]

    final_results.append(summary)

# 合併所有類別的統計結果
from functools import reduce
click_stats = reduce(lambda left, right: pd.merge(left, right, on=['code_module', 'code_presentation', 'id_student'], how='outer'), final_results)
click_stats.head()


/var/folders/_q/xfvw_6_d3c5b8t1lpm7rppnh0000gn/T/ipykernel_7823/371790850.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_full = df_merged.groupby(['code_module', 'code_presentation', 'id_student']).apply(create_full_date_range).reset_index(drop=True)
/var/folders/_q/xfvw_6_d3c5b8t1lpm7rppnh0000gn/T/ipykernel_7823/371790850.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_full = df_merged.gro

,code_module,code_presentation,id_student,all_mean_click_interval,all_std_click_interval,all_mean_clicks,all_click_variance,lpb_mean_click_interval,lpb_std_click_interval,lpb_mean_clicks,...,kab_mean_clicks,kab_click_variance,ilb_mean_click_interval,ilb_std_click_interval,ilb_mean_clicks,ilb_click_variance,lcb_mean_click_interval,lcb_std_click_interval,lcb_mean_clicks,lcb_click_variance
0,AAA,2013J,11391,6.615385,5.756531,23.350000,1.176466,6.615385,5.756531,4.250000,...,22.840000,1.177013,13.888889,14.220486,10.157895,0.813224,NaN,NaN,NaN,NaN
1,AAA,2013J,28400,3.151899,2.931214,17.937500,1.143786,3.151899,2.931214,5.137500,...,12.914894,1.101394,5.318182,4.397530,9.266667,0.823859,NaN,NaN,NaN,NaN
2,AAA,2013J,30268,2.000000,1.612452,23.416667,0.705049,2.000000,1.612452,6.750000,...,10.571429,0.768252,2.750000,1.581139,14.000000,0.617213,NaN,NaN,NaN,NaN
3,AAA,2013J,31604,2.245902,1.895238,17.544715,0.863160,2.283333,1.928047,4.760331,...,10.533333,0.986124,3.685714,3.019468,8.929577,0.775262,NaN,NaN,NaN,NaN
4,AAA,2013J,32885,3.724638,3.883919,14.771429,1.116342,3.835821,4.024911,4.161765,...,9.603448,1.055193,10.708333,22.651383,7.760000,0.845644,NaN,NaN,NaN,NaN


In [21]:
merged_df = pd.merge(merged_df, click_stats, on=['code_module', 'code_presentation', 'id_student'], how='left')
print(merged_df.shape)

(15961, 41)


## 不同學習行為類型的比例

In [22]:
import pandas as pd

# 假設 df 是學生成績和行為的資料，vle 是提供資源的資料
# 讀取資料
df = pd.read_csv('studentVle.csv')
vle = pd.read_csv('vle.csv')
vle = vle.merge(df, on=['id_site', "code_module","code_presentation"], how='inner')

# 1. 計算每堂課提供的各資源類型比例
# 首先，將每個 id_site 與對應的資源類型對應起來
vle['category'] = vle['activity_type'].map(activity_to_category)

# 計算每堂課的各資源類型的點擊數總和
resource_proportions = vle.groupby(['code_module', 'code_presentation', 'category']).agg(
    total_clicks=('sum_click', 'sum')
).reset_index()

# 計算每堂課總點擊數
total_clicks_per_class = resource_proportions.groupby(['code_module', 'code_presentation'])['total_clicks'].sum().reset_index()
total_clicks_per_class.rename(columns={'total_clicks': 'class_total_clicks'}, inplace=True)

# 合併每堂課的總點擊數到資源比例資料中
resource_proportions = pd.merge(resource_proportions, total_clicks_per_class, on=['code_module', 'code_presentation'])

# 計算每堂課每類資源的比例
resource_proportions['resource_ratio'] = resource_proportions['total_clicks'] / resource_proportions['class_total_clicks']

# 2. 計算每位學生對不同資源類型的點擊比例
student_resource_proportions = vle.groupby(['id_student', 'code_module', 'code_presentation', 'category']).agg(
    student_clicks=('sum_click', 'sum')
).reset_index()

# 計算每個學生在每堂課的總點擊數
student_total_clicks = student_resource_proportions.groupby(['id_student', 'code_module', 'code_presentation'])['student_clicks'].sum().reset_index()
student_total_clicks.rename(columns={'student_clicks': 'student_total_clicks'}, inplace=True)

# 合併每個學生的總點擊數到學生資源比例資料中
student_resource_proportions = pd.merge(student_resource_proportions, student_total_clicks, on=['id_student', 'code_module', 'code_presentation'])

# 計算每位學生對不同資源類型的比例
student_resource_proportions['student_resource_ratio'] = student_resource_proportions['student_clicks'] / student_resource_proportions['student_total_clicks']

# 3. 計算每位學生對每類資源的點擊比例與該班級提供資源比例的對比
final_result = pd.merge(student_resource_proportions, resource_proportions[['code_module', 'code_presentation', 'category', 'resource_ratio']], 
                        on=['code_module', 'code_presentation', 'category'])

# 計算可比較的比例（學生比例 / 該班級的比例）
final_result['normalized_resource_ratio'] = final_result['student_resource_ratio'] / final_result['resource_ratio']

# 4. 將結果轉換為寬格式
# 先做 pivot
wide = final_result.pivot_table(
    index=['code_module', 'code_presentation', 'id_student'],
    columns='category',
    values=['student_resource_ratio',], # 'resource_ratio', 'normalized_resource_ratio'
    fill_value=0
)

# 把 MultiIndex 的欄位壓平，改成 "KAB_resource_ratio" 這種形式
wide.columns = [
    f"{cat}_{metric}"
    for metric, cat in wide.columns
]
wide = wide.reset_index()

In [23]:
merged_df = pd.merge(merged_df, wide, on=['code_module', 'code_presentation', 'id_student'], how='left')
print(merged_df.shape)

(15961, 45)


# 標準化差值

In [24]:
# # 讀取資料
# student_assessments = pd.read_csv('studentAssessment.csv')
# assessments = pd.read_csv('assessments.csv')
# assessments = assessments[assessments['assessment_type'] != 'Exam']
# merged_data = pd.merge(student_assessments, assessments, on='id_assessment')

# merged_data['z_score'] = merged_data.groupby(['code_module', 'code_presentation'])['score'].transform(lambda x: (x - x.mean()) / x.std())

# # 對每位學生在每班的作業依日期排序
# merged_data = merged_data.sort_values(by=['code_module', 'code_presentation', 'id_student', 'date'])

# # 4. 找出每個學生在每班的第一筆與最後一筆作業分數（z_score）
# exam_data = merged_data.groupby(['code_module', 'code_presentation', 'id_student']).agg(
#     first_score=('score', 'first'),
#     last_score=('score', 'last')
# ).reset_index()

# # 5. 計算成績差異
# exam_data['score_diff'] = exam_data['last_score'] - exam_data['first_score']

# # 依照 期末考分數 分成五群
# # exam_data = merged_data[merged_data['assessment_type'] == 'Exam'].copy()  # 使用 copy() 創建副本
# exam_data = exam_data.dropna(subset=['score_diff'])
# # exam_data['score'] = pd.to_numeric(exam_data['score'], errors='coerce')
# exam_data_sorted = exam_data.sort_values(by='score_diff', ascending=False)
# exam_data.drop(columns=['first_score', 'last_score'], inplace=True)

In [25]:
# merged_df = pd.merge(merged_df, exam_data, on=['code_module', 'code_presentation', 'id_student'], how='left')
# print(merged_df.shape)

# 匯出

In [26]:
df = normalize(merged_df)
df.to_csv(export_name, index=False)

         id_student   early_score    late_score     mid_score    score_diff  \
count  1.596100e+04  15961.000000  15961.000000  15460.000000  15961.000000   
mean   7.134336e+05      0.766354      0.743110      0.739054     -2.324383   
std    5.617378e+05      0.143780      0.171068      0.151005     17.810969   
min    6.516000e+03      0.000000      0.000000      0.000000    -86.666667   
25%    5.054130e+05      0.700000      0.645000      0.650000    -12.666667   
50%    5.871920e+05      0.796667      0.777143      0.765000     -2.000000   
75%    6.450470e+05      0.866667      0.870000      0.855000      7.571429   
max    2.698588e+06      1.000000      1.000000      1.000000     76.666667   

       num_of_prev_attempts  studied_credits  date_registration  \
count          15961.000000     15961.000000       15961.000000   
mean               0.022565         0.078981           0.589735   
std                0.071587         0.061642           0.115441   
min                0